# Example 01.02 — upload and attach datasets

Uploads three deterministic datasets: training data, an unlabeled batch-scoring
cohort, and delayed actuals. Existing attached families are reused without creating
duplicate immutable versions.

Prerequisite: run `Example01_01_setup_business_case.ipynb`.


In [ ]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "ml_app_client").is_dir()),
    None,
)
if REPOSITORY_ROOT is None:
    raise RuntimeError("Start Jupyter inside the ml-app repository")
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from ml_app_client import MLAppClient, ResourceNotFoundError

client = MLAppClient.connect()
print("Connected to ML App")


## Choose resource names

These are normal names passed to `ml_app_client`. Edit them directly and use the
same values in the following notebooks. Dataset and pipeline names are resolved
inside the selected Business Case. Business Case and service names are globally
unique on one ML App installation.


In [ ]:
# These are ordinary platform resource names. Change the two globally unique
# names (Business Case and model service) when sharing one installation.
BUSINESS_CASE_NAME = "[MLAPP EXAMPLE 01 v2] Estates Lifecycle - demo"
TRAINING_DATASET_NAME = "Example01 Estates - Training"
SCORING_DATASET_NAME = "Example01 Estates - Batch Input"
ACTUALS_DATASET_NAME = "Example01 Estates - Actuals"
TRAINING_PIPELINE_NAME = "Example01 03 - AutoML Training"
BATCH_PIPELINE_NAME = "Example01 05 - Batch Scoring"
MONITORING_PIPELINE_NAME = "Example01 07 - Performance Monitoring"
MODEL_NAME = "Example01 Estates Price Model"
OUTPUT_NAME_PREFIX = "Example01 Estates AutoML"
MODEL_SERVICE_NAME = "Example01 10 - Estates Model Service - demo"

TRAINING_RUN_KEY = "Example01-training-v2"
BATCH_RUN_KEY = "Example01-batch-scoring-v2"
MONITORING_RUN_KEY = "Example01-monitoring-v2"

print("Resource names configured for:", BUSINESS_CASE_NAME)


## 1. Describe the datasets we need


In [ ]:
from examples.example01_lifecycle import SCENARIO_TAGS, data_file

specifications = [
    {"name": TRAINING_DATASET_NAME, "file": "regression-example.csv", "role": "source", "row_id": "property_id", "target": "sale_price_pln"},
    {"name": SCORING_DATASET_NAME, "file": "estates-sale-prices-batch-scoring-100k.parquet", "role": "scoring_input", "row_id": "property_id", "target": ""},
    {"name": ACTUALS_DATASET_NAME, "file": "estates-sale-prices-batch-scoring-100k-actuals.parquet", "role": "monitoring_actuals", "row_id": "property_id", "target": "sale_price_pln"},
]
business_case = client.business_case_by_name(BUSINESS_CASE_NAME)


## 2. Check which datasets already exist in this Business Case


In [ ]:
datasets = {}
missing = []
for specification in specifications:
    try:
        dataset = client.dataset_by_name(
            business_case_name=BUSINESS_CASE_NAME,
            dataset_name=specification["name"],
        )
        datasets[specification["name"]] = dataset
        print(f"FOUND {dataset.name} v{dataset.version_number}; rows={dataset.row_count:,}")
    except ResourceNotFoundError:
        missing.append(specification)
        print(f"NOT FOUND {specification['name']}")


## 3. Upload every missing file


In [ ]:
for specification in missing:
    dataset = client.upload_dataset(
        data_file(specification["file"]),
        name=specification["name"],
        description=f"Deterministic Example 01 dataset: {specification['role']}",
        tags=SCENARIO_TAGS,
    )
    datasets[specification["name"]] = dataset
    print(f"UPLOADED {dataset.name} v{dataset.version_number}; rows={dataset.row_count:,}")


## 4. Attach newly uploaded datasets to the Business Case


In [ ]:
for specification in missing:
    dataset = datasets[specification["name"]]
    client.attach_dataset(
        str(business_case["id"]),
        dataset.id,
        role=specification["role"],
        context_note="Example01 contract v1.0",
        primary_key_column=specification["row_id"],
        target_column=specification["target"],
    )
    print(f"ATTACHED {dataset.name} as {specification['role']}")

if not missing:
    print("Nothing to upload or attach")


All three operations stream files or bounded metadata; no full dataset is loaded into notebook memory.
